# Robust Heat-Flux Validation Framework

This notebook audits and extends the current parametric PINN evaluation pipeline.

The emphasis is deliberately **heat-flux prediction**, not temperature reconstruction.

It adds:

- case-level train / validation / test / OOD splits
- leave-one-flux-family-out analysis
- flux-first metrics
- noise robustness campaigns
- sensor sensitivity studies
- physics consistency checks
- baseline regression comparison
- support for multi-checkpoint / multi-seed aggregation


## Audit Summary

### Weaknesses found in the current codebase

- Existing validation notebooks treat `theta` RMSE as the main success criterion.
- Validation is mostly notebook-local and not modularized for repeated campaigns.
- `manifest_testing.csv` is used as a new-case pool, but there is no explicit separation between in-distribution held-out test cases and OOD cases.
- There is no built-in noise campaign, sensor study, or uncertainty aggregation.
- There is no leave-one-flux-family-out evaluation.
- There is no baseline inverse model comparison in the validation loop.
- Physics checks are logged weakly during training, but not treated as first-class validation outputs.

### Design intent of this notebook

This notebook uses `src.robust_validation` to make validation reproducible, heat-flux-centered, and extensible.


## Setup


In [43]:
import sys
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Could not find project root containing /src")
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.utils import get_device, set_seed, ensure_dir
from src.pinn import LeftBoundaryAnsatzMLP
from src.robust_validation import (
    SplitBundle,
    ObservationConfig,
    NoiseConfig,
    build_case_splits,
    bundle_metrics_summary,
    compute_mu_stats,
    evaluate_cases,
    fit_linear_flux_baseline,
    leave_one_flux_family_out,
    load_cases_with_mu,
    predict_linear_flux_baseline,
    results_to_frame,
    run_noise_campaign,
    run_sensor_study,
    save_experiment_log,
)

set_seed(42)
device = get_device()
print(f"Project root: {ROOT}")
print(f"Device: {device}")

PART = "2026_03_17_1034"
MODEL_ROOT = ROOT / "outputs" / "parametric" / PART

MODEL_SPECS = [
    {
        "name": "hard_left_bc_seed0",
        "checkpoint": MODEL_ROOT / "model" / "best.pt",
    },
]

MANIFEST_TRAIN = ROOT / "data" / "manifest.csv"
MANIFEST_TESTING = ROOT / "data" / "manifest_testing.csv"
MU_TIME_SAMPLES = 15

RUN_ID = datetime.now(timezone.utc).strftime("%Y_%m_%d_%H%M")
OUTDIR = ensure_dir(MODEL_ROOT / "robust_validation" / RUN_ID)
print(f"Output directory: {OUTDIR}")


Project root: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN
Device: cpu
Output directory: c:\Users\wscm13\OneDrive - Loughborough University\Part C\IDP\Individual Project\PINN\outputs\parametric\2026_03_17_1034\robust_validation\2026_03_17_1453


## Split Cases by Case ID and OOD Rules


In [44]:
splits = build_case_splits(MANIFEST_TRAIN, MANIFEST_TESTING, val_fraction=0.2)
print("Train cases:", [r["case_id"] for r in splits.train_rows])
print("Held-out cases:", [r["case_id"] for r in splits.val_rows])
print("Test cases:", [r["case_id"] for r in splits.test_rows])
print("OOD cases:", [r["case_id"] for r in splits.ood_rows])

loo_splits = leave_one_flux_family_out(splits.train_rows + splits.val_rows)
print("Leave-one-family-out splits:", list(loo_splits.keys()))


Train cases: ['const_10000', 'const_10000_Tleft_280', 'const_10000_Tleft_300', 'const_10000_Tleft_310', 'const_12000', 'const_14000', 'const_15000', 'const_20000', 'const_5000', 'const_6000', 'offset_sine_1', 'sine_A10000_T100', 'sine_A10000_T50', 'sine_A12500_T125', 'sine_A12500_T25', 'sine_A12500_T75', 'sine_A5000_T100']
Held-out cases: ['sine_A5000_T50', 'sine_A7500_T125', 'sine_A7500_T25', 'sine_A7500_T75']
Test cases: ['const_6000', 'const_12000', 'const_14000', 'const_20000', 'sine_A5000_T50', 'sine_A7500_T75', 'sine_A7500_T125', 'sine_A7500_T25', 'sine_A12500_T75', 'sine_A12500_T125', 'sine_A12500_T25', 'const_10000_Tleft_280', 'const_10000_Tleft_300', 'const_10000_Tleft_310']
OOD cases: []
Leave-one-family-out splits: ['constant', 'offset_sine', 'sine']


## Load Cases and Compute Training-Normalized Conditioning


In [45]:
train_cases = load_cases_with_mu(splits.train_rows, root=ROOT, mu_time_samples=MU_TIME_SAMPLES)
val_cases = load_cases_with_mu(splits.val_rows, root=ROOT, mu_time_samples=MU_TIME_SAMPLES)
test_cases = load_cases_with_mu(splits.test_rows, root=ROOT, mu_time_samples=MU_TIME_SAMPLES)
ood_cases = load_cases_with_mu(splits.ood_rows, root=ROOT, mu_time_samples=MU_TIME_SAMPLES)

mu_stats = compute_mu_stats(train_cases)
for case_group in [train_cases, val_cases, test_cases, ood_cases]:
    for case in case_group:
        case["mu"] = 2.0 * (case["mu_raw"] - mu_stats["mu_min"]) / (mu_stats["mu_max"] - mu_stats["mu_min"] + mu_stats["eps"]) - 1.0

print(f"Loaded {len(train_cases)} train cases")
print(f"Loaded {len(val_cases)} held-out cases")
print(f"Loaded {len(test_cases)} test cases")
print(f"Loaded {len(ood_cases)} OOD cases")


Loaded 17 train cases
Loaded 4 held-out cases
Loaded 14 test cases
Loaded 0 OOD cases


## Load Checkpoints


In [46]:
def load_model(spec):
    ckpt_path = Path(spec["checkpoint"])
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location=device)
    mu_dim = checkpoint.get("mu_dim", 30)
    model = LeftBoundaryAnsatzMLP(in_dim=2 + mu_dim, hidden=64, layers=4).to(device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model, checkpoint

loaded_models = {}
for spec in MODEL_SPECS:
    model, checkpoint = load_model(spec)
    loaded_models[spec["name"]] = {
        "model": model,
        "checkpoint": checkpoint,
        "path": spec["checkpoint"],
    }

print("Loaded models:", list(loaded_models.keys()))


Loaded models: ['hard_left_bc_seed0']


## Clean Evaluation Configuration


In [47]:
clean_obs = ObservationConfig(
    sensor_indices=None,
    sensor_mode="nearest_right",
    time_stride=1,
    mu_time_samples=MU_TIME_SAMPLES,
)
clean_noise = NoiseConfig(name="clean")


## Clean Flux-First Evaluation


In [48]:
all_frames = []
all_results = {}

for model_name, payload in loaded_models.items():
    model = payload["model"]
    split_map = {
        "val": val_cases,
        "test": test_cases,
        "ood": ood_cases,
    }
    all_results[model_name] = {}
    for split_name, cases in split_map.items():
        if not cases:
            continue
        results = evaluate_cases(
            model,
            cases,
            mu_stats,
            device,
            obs_cfg=clean_obs,
            noise_cfg=clean_noise,
            seed=42,
        )
        all_results[model_name][split_name] = results
        frame = results_to_frame(results, split_name=split_name, model_name=model_name)
        all_frames.append(frame)

clean_metrics_df = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
display(clean_metrics_df)

clean_summary_df = bundle_metrics_summary(clean_metrics_df, ["model_name", "split", "noise_name"]) if not clean_metrics_df.empty else pd.DataFrame()
display(clean_summary_df)


KeyboardInterrupt: 

## Baseline Regression Comparison


In [ ]:
baseline_rows = []
if loaded_models:
    baseline_model_name = next(iter(loaded_models))
    train_results_for_baseline = evaluate_cases(
        loaded_models[baseline_model_name]["model"],
        train_cases,
        mu_stats,
        device,
        obs_cfg=clean_obs,
        noise_cfg=clean_noise,
        seed=100,
    )
    train_predictions = [r["prediction"] for r in train_results_for_baseline]
    baseline = fit_linear_flux_baseline(train_predictions)

    for split_name, cases in {"test": test_cases, "ood": ood_cases}.items():
        for case in cases:
            mu = case["mu"]
            q_true = case["raw"]["q_right"].reshape(-1)
            q_pred = predict_linear_flux_baseline(baseline, mu)
            diff = q_pred - q_true
            baseline_rows.append({
                "case_id": case["case_id"],
                "split": split_name,
                "model_name": "linear_baseline",
                "noise_name": "clean",
                "flux_rmse": float(np.sqrt(np.mean(diff ** 2))),
                "flux_mae": float(np.mean(np.abs(diff))),
                "flux_rel_l2": float(np.linalg.norm(diff) / (np.linalg.norm(q_true) + 1e-12)),
                "flux_max_abs": float(np.max(np.abs(diff))),
            })

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df)
if not baseline_df.empty and not clean_metrics_df.empty:
    baseline_compare_df = pd.concat([
        clean_metrics_df[[c for c in clean_metrics_df.columns if c in baseline_df.columns]],
        baseline_df,
    ], ignore_index=True)
    display(baseline_compare_df)


## Example Flux and Residual Plots


In [ ]:
def get_first_prediction(results_dict, split_name):
    split_results = results_dict.get(split_name, [])
    if not split_results:
        return None
    return split_results[0]["prediction"]

example_model = next(iter(all_results)) if all_results else None
example_pred = get_first_prediction(all_results[example_model], "test") if example_model else None
if example_pred is None:
    print("No example prediction available.")
else:
    tau = example_pred["tau"]
    xi = example_pred["xi"]
    q_true = example_pred["q_true"]
    q_pred = example_pred["q_pred"]
    residual = q_pred - q_true

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

    axes[0, 0].plot(tau, q_true, 'k-', linewidth=2, label='True flux')
    axes[0, 0].plot(tau, q_pred, 'r--', linewidth=2, label='Predicted flux')
    axes[0, 0].set_title(f'Flux vs Time: {example_pred["case_id"]}')
    axes[0, 0].set_xlabel('tau')
    axes[0, 0].set_ylabel('q')
    axes[0, 0].grid(True)
    axes[0, 0].legend()

    axes[0, 1].scatter(q_true, q_pred, s=12, alpha=0.7)
    mn = min(float(q_true.min()), float(q_pred.min()))
    mx = max(float(q_true.max()), float(q_pred.max()))
    axes[0, 1].plot([mn, mx], [mn, mx], 'k--', linewidth=1)
    axes[0, 1].set_title('Parity Plot (Flux)')
    axes[0, 1].set_xlabel('True q')
    axes[0, 1].set_ylabel('Predicted q')
    axes[0, 1].grid(True)

    axes[1, 0].plot(tau, residual, color='tab:purple', linewidth=1.5)
    axes[1, 0].axhline(0.0, color='k', linestyle='--', linewidth=1)
    axes[1, 0].set_title('Flux Residual vs Time')
    axes[1, 0].set_xlabel('tau')
    axes[1, 0].set_ylabel('q_pred - q_true')
    axes[1, 0].grid(True)

    field_error = example_pred["T_pred"] - example_pred["T_true"]
    err_lim = np.max(np.abs(field_error))
    im = axes[1, 1].imshow(
        field_error,
        aspect='auto',
        origin='lower',
        extent=[tau.min(), tau.max(), xi.min(), xi.max()],
        cmap='coolwarm',
        vmin=-err_lim,
        vmax=err_lim,
    )
    axes[1, 1].set_title('Space-Time Temperature Error Heatmap')
    axes[1, 1].set_xlabel('tau')
    axes[1, 1].set_ylabel('xi')
    plt.colorbar(im, ax=axes[1, 1])

    example_plot_path = OUTDIR / f"example_flux_validation_{example_pred['case_id']}.png"
    plt.savefig(example_plot_path, dpi=150, bbox_inches='tight')
    print(f"Saved example plot to: {example_plot_path}")
    plt.show()


## Noise Robustness Campaign


In [ ]:
noise_scenarios = [
    NoiseConfig(name="gaussian_0p1pct", gaussian_std=0.001),
    NoiseConfig(name="gaussian_0p5pct", gaussian_std=0.005),
    NoiseConfig(name="gaussian_1pct", gaussian_std=0.01),
    NoiseConfig(name="gaussian_2pct", gaussian_std=0.02),
    NoiseConfig(name="gaussian_5pct", gaussian_std=0.05),
    NoiseConfig(name="bias_1pct", constant_bias=0.01),
    NoiseConfig(name="drift_2pct", drift_final=0.02),
    NoiseConfig(name="sensor_specific", gaussian_std=0.01, sensor_noise_std_scale=0.5),
    NoiseConfig(name="temporal_corr", gaussian_std=0.01, temporal_corr=4),
    NoiseConfig(name="spatial_corr", gaussian_std=0.01, spatial_corr=2),
    NoiseConfig(name="spikes", gaussian_std=0.005, spike_fraction=0.01, spike_scale=0.2),
    NoiseConfig(name="missing_intervals", gaussian_std=0.005, missing_fraction=0.15, missing_axis="time"),
    NoiseConfig(name="dead_sensors", gaussian_std=0.005, dead_sensor_fraction=0.2),
]

noise_frames = []
for model_name, payload in loaded_models.items():
    model = payload["model"]
    campaign_df = run_noise_campaign(
        model,
        test_cases,
        mu_stats,
        device,
        obs_cfg=clean_obs,
        noise_scenarios=noise_scenarios,
        base_seed=123,
    )
    if campaign_df.empty:
        continue
    campaign_df["model_name"] = model_name
    noise_frames.append(campaign_df)

noise_df = pd.concat(noise_frames, ignore_index=True) if noise_frames else pd.DataFrame()
display(noise_df)

noise_summary_df = bundle_metrics_summary(noise_df, ["model_name", "noise_name"]) if not noise_df.empty else pd.DataFrame()
display(noise_summary_df)


## Noise Degradation Curves


In [ ]:
if noise_df.empty:
    print("Noise campaign has no results.")
else:
    gaussian_df = noise_df[noise_df["noise_name"].str.startswith("gaussian_")].copy()
    if gaussian_df.empty:
        print("No Gaussian scenarios found.")
    else:
        order = {
            "gaussian_0p1pct": 0.1,
            "gaussian_0p5pct": 0.5,
            "gaussian_1pct": 1.0,
            "gaussian_2pct": 2.0,
            "gaussian_5pct": 5.0,
        }
        gaussian_df["noise_level_pct"] = gaussian_df["noise_name"].map(order)
        curve_df = gaussian_df.groupby(["model_name", "noise_level_pct"], as_index=False)["flux_rmse"].mean()
        plt.figure(figsize=(8, 4))
        for model_name, grp in curve_df.groupby("model_name"):
            grp = grp.sort_values("noise_level_pct")
            plt.plot(grp["noise_level_pct"], grp["flux_rmse"], marker='o', label=model_name)
        plt.xlabel('Gaussian noise level (%)')
        plt.ylabel('Mean flux RMSE')
        plt.title('Heat-Flux Error vs Gaussian Noise Level')
        plt.grid(True)
        plt.legend()
        noise_curve_path = OUTDIR / 'flux_rmse_vs_gaussian_noise.png'
        plt.savefig(noise_curve_path, dpi=150, bbox_inches='tight')
        print(f"Saved noise curve to: {noise_curve_path}")
        plt.show()


## Sensor Sensitivity Study


In [ ]:
sensor_modes = [
    ("all", None),
    ("random", 8),
    ("random", 4),
    ("boundary_only", 2),
    ("interior_only", 4),
]

sensor_frames = []
for model_name, payload in loaded_models.items():
    study_df = run_sensor_study(
        payload["model"],
        test_cases,
        mu_stats,
        device,
        sensor_modes=sensor_modes,
        noise_cfg=clean_noise,
        mu_time_samples=MU_TIME_SAMPLES,
        base_seed=222,
    )
    if study_df.empty:
        continue
    study_df["model_name"] = model_name
    sensor_frames.append(study_df)

sensor_df = pd.concat(sensor_frames, ignore_index=True) if sensor_frames else pd.DataFrame()
display(sensor_df)


## Sensor Count Curves


In [ ]:
if sensor_df.empty:
    print("Sensor study has no results.")
else:
    curve_df = sensor_df.groupby(["model_name", "sensor_count"], as_index=False)["flux_rmse"].mean()
    plt.figure(figsize=(8, 4))
    for model_name, grp in curve_df.groupby("model_name"):
        grp = grp.sort_values("sensor_count")
        plt.plot(grp["sensor_count"], grp["flux_rmse"], marker='o', label=model_name)
    plt.xlabel('Sensor count used to build observation trace')
    plt.ylabel('Mean flux RMSE')
    plt.title('Heat-Flux Error vs Sensor Count')
    plt.grid(True)
    plt.legend()
    sensor_curve_path = OUTDIR / 'flux_rmse_vs_sensor_count.png'
    plt.savefig(sensor_curve_path, dpi=150, bbox_inches='tight')
    print(f"Saved sensor curve to: {sensor_curve_path}")
    plt.show()


## Multi-Checkpoint / Multi-Seed Aggregation


In [ ]:
seed_rows = []
for model_name, split_results in all_results.items():
    for split_name, results in split_results.items():
        for result in results:
            seed_rows.append({
                "model_name": model_name,
                "split": split_name,
                "case_id": result["prediction"]["case_id"],
                "flux_rmse": result["metrics"]["flux_rmse"],
            })
seed_df = pd.DataFrame(seed_rows)
display(seed_df)

if seed_df["model_name"].nunique() > 1:
    plt.figure(figsize=(8, 4))
    seed_df.boxplot(column="flux_rmse", by="model_name", rot=20)
    plt.suptitle('')
    plt.title('Flux RMSE Distribution Across Checkpoints / Seeds')
    plt.ylabel('Flux RMSE')
    seed_boxplot_path = OUTDIR / 'seed_variability_boxplot.png'
    plt.savefig(seed_boxplot_path, dpi=150, bbox_inches='tight')
    print(f"Saved seed variability boxplot to: {seed_boxplot_path}")
    plt.show()
else:
    print("Add more checkpoints to MODEL_SPECS to activate seed / ensemble analysis.")


## Leave-One-Flux-Family-Out Evaluation Scaffold


In [ ]:
loo_rows = []
for family, loo in loo_splits.items():
    loo_train_cases = load_cases_with_mu(loo.train_rows, root=ROOT, mu_time_samples=MU_TIME_SAMPLES)
    loo_test_cases = load_cases_with_mu(loo.test_rows, root=ROOT, mu_time_samples=MU_TIME_SAMPLES)
    if not loo_train_cases or not loo_test_cases:
        continue
    loo_mu_stats = compute_mu_stats(loo_train_cases)
    for case_group in [loo_train_cases, loo_test_cases]:
        for case in case_group:
            case["mu"] = 2.0 * (case["mu_raw"] - loo_mu_stats["mu_min"]) / (loo_mu_stats["mu_max"] - loo_mu_stats["mu_min"] + loo_mu_stats["eps"]) - 1.0

    model_name = next(iter(loaded_models))
    results = evaluate_cases(
        loaded_models[model_name]["model"],
        loo_test_cases,
        loo_mu_stats,
        device,
        obs_cfg=clean_obs,
        noise_cfg=clean_noise,
        seed=999,
    )
    frame = results_to_frame(results, split_name=f"loo_{family}", model_name=model_name)
    frame["held_out_family"] = family
    loo_rows.append(frame)

loo_df = pd.concat(loo_rows, ignore_index=True) if loo_rows else pd.DataFrame()
display(loo_df)


## Save Tables and Config


In [ ]:
artifacts = {
    "clean_metrics": clean_metrics_df,
    "clean_summary": clean_summary_df,
    "baseline": baseline_df if 'baseline_df' in globals() else pd.DataFrame(),
    "noise": noise_df,
    "noise_summary": noise_summary_df if 'noise_summary_df' in globals() else pd.DataFrame(),
    "sensor": sensor_df,
    "seed": seed_df,
    "loo": loo_df,
}
for name, frame in artifacts.items():
    if isinstance(frame, pd.DataFrame) and not frame.empty:
        csv_path = OUTDIR / f"{name}.csv"
        frame.to_csv(csv_path, index=False)
        print(f"Saved {name} table to: {csv_path}")

config_payload = {
    "model_specs": [{"name": spec["name"], "checkpoint": str(spec["checkpoint"])} for spec in MODEL_SPECS],
    "manifest_train": str(MANIFEST_TRAIN),
    "manifest_testing": str(MANIFEST_TESTING),
    "mu_time_samples": MU_TIME_SAMPLES,
    "output_dir": str(OUTDIR),
    "noise_scenarios": [scenario.__dict__ for scenario in noise_scenarios],
    "sensor_modes": sensor_modes,
}
save_experiment_log(OUTDIR, "robust_validation_config.json", config_payload)
print(f"Saved config to: {OUTDIR / 'robust_validation_config.json'}")
